# GridPulse UK — Silver Carbon-Intensity Transformation

This notebook converts immutable Bronze snapshots into a clean and trusted
Silver dataset.

The transformation logic lives in `src/gridpulse/transformations.py`, which is
covered by unit tests in `tests/` and run by GitHub Actions on every pull request.
This notebook calls those tested functions and handles reading, validation
checks and writing.

## Silver rules

1. Reject structurally invalid records (quarantine table, with a reason).
2. Allow actual carbon intensity to be null for unfinished periods.
3. Group repeated snapshots by electricity period.
4. Prefer a snapshot containing an actual value.
5. Otherwise, use the most recently ingested snapshot.
6. Calculate forecast-error metrics.
7. Maintain one canonical Silver record per electricity period.


In [ ]:
import os
import sys

# Make the repo's src/ folder importable from this notebook (notebooks/ -> ../src)
sys.path.append(os.path.abspath("../src"))

from pyspark.sql.functions import col
from gridpulse.transformations import (
    split_valid_rejected,
    select_best_snapshot,
    to_silver_record,
)

# If you edit transformations.py later, restart Python first so the change is picked up:
# dbutils.library.restartPython()

In [ ]:
BRONZE_TABLE = "gridpulse.bronze.carbon_intensity"
SILVER_TABLE = "gridpulse.silver.carbon_intensity"
REJECTED_TABLE = "gridpulse.silver.carbon_intensity_rejected"

bronze_source_df = spark.table(BRONZE_TABLE)

print(
    f"Bronze records available: "
    f"{bronze_source_df.count()}"
)

bronze_source_df.printSchema()

In [ ]:
# Apply Silver data-quality rules (tested in tests/test_transformations.py)
valid_bronze_df, rejected_bronze_df = split_valid_rejected(bronze_source_df)

valid_count = valid_bronze_df.count()
rejected_count = rejected_bronze_df.count()

print(f"Valid Bronze records: {valid_count}")
print(f"Rejected Bronze records: {rejected_count}")

In [ ]:
from pyspark.sql.functions import count

repeated_periods_df = (
    valid_bronze_df
    .groupBy(
        "period_start_utc",
        "period_end_utc"
    )
    .agg(
        count("*").alias("snapshot_count")
    )
    .filter(
        col("snapshot_count") > 1
    )
    .orderBy(
        col("snapshot_count").desc(),
        col("period_start_utc")
    )
)

display(repeated_periods_df)

In [ ]:
# One canonical snapshot per electricity period:
# prefer a snapshot with an actual value, then the newest ingestion
canonical_periods_df = select_best_snapshot(valid_bronze_df)

canonical_count = canonical_periods_df.count()

print(
    f"Canonical electricity periods: "
    f"{canonical_count}"
)

In [ ]:
# Shape into Silver columns: period_id, forecast error, lineage, processing time
silver_ready_df = to_silver_record(canonical_periods_df)

display(
    silver_ready_df
    .orderBy("period_start_utc")
    .limit(10)
)

In [ ]:
silver_count = silver_ready_df.count()

distinct_period_count = (
    silver_ready_df
    .select("period_id")
    .distinct()
    .count()
)

duplicate_period_ids = (
    silver_count
    - distinct_period_count
)

missing_period_ids = (
    silver_ready_df
    .filter(col("period_id").isNull())
    .count()
)

print(f"Silver rows: {silver_count}")
print(f"Distinct period IDs: {distinct_period_count}")
print(f"Duplicate period IDs: {duplicate_period_ids}")
print(f"Missing period IDs: {missing_period_ids}")

assert silver_count > 0, (
    "Silver contains no records."
)

assert duplicate_period_ids == 0, (
    "Silver contains duplicate period IDs."
)

assert missing_period_ids == 0, (
    "Silver contains missing period IDs."
)

print("Silver in-memory validation passed.")

In [ ]:
from delta.tables import DeltaTable

if not spark.catalog.tableExists(SILVER_TABLE):

    (
        silver_ready_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(SILVER_TABLE)
    )

    print(f"Created Silver table: {SILVER_TABLE}")

else:

    silver_delta_table = DeltaTable.forName(
        spark,
        SILVER_TABLE
    )

    (
        silver_delta_table.alias("target")
        .merge(
            silver_ready_df.alias("source"),
            "target.period_id = source.period_id"
        )
        .whenMatchedUpdateAll(
            condition=(
                "target.source_record_id "
                "<> source.source_record_id"
            )
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(f"Updated Silver table: {SILVER_TABLE}")

In [ ]:
if not spark.catalog.tableExists(REJECTED_TABLE):

    (
        rejected_bronze_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(REJECTED_TABLE)
    )

    print(
        f"Created rejected-record table: "
        f"{REJECTED_TABLE}"
    )

elif rejected_count > 0:

    rejected_delta_table = DeltaTable.forName(
        spark,
        REJECTED_TABLE
    )

    (
        rejected_delta_table.alias("target")
        .merge(
            rejected_bronze_df.alias("source"),
            "target.record_id = source.record_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(
        f"Merged rejected records into: "
        f"{REJECTED_TABLE}"
    )

else:

    print(
        "No rejected records to merge. "
        "The rejected-record table already exists."
    )

In [ ]:
%sql
COMMENT ON TABLE gridpulse.silver.carbon_intensity IS
'Canonical validated national carbon-intensity record for each GB electricity period';

ALTER TABLE gridpulse.silver.carbon_intensity
SET TBLPROPERTIES (
    'quality' = 'silver',
    'source.system' = 'NESO Carbon Intensity API',
    'data.owner' = 'GridPulse',
    'business.key' = 'period_start_utc, period_end_utc'
);

In [ ]:
%sql
COMMENT ON TABLE gridpulse.silver.carbon_intensity_rejected IS
'Bronze carbon-intensity records rejected by Silver data-quality rules';

ALTER TABLE gridpulse.silver.carbon_intensity_rejected
SET TBLPROPERTIES (
    'quality' = 'quarantine',
    'source.system' = 'NESO Carbon Intensity API',
    'data.owner' = 'GridPulse'
);

In [ ]:
%sql
SELECT
    COUNT(*) AS silver_rows,
    COUNT(DISTINCT period_id) AS distinct_period_ids,
    COUNT(*) - COUNT(DISTINCT period_id) AS duplicate_period_ids
FROM gridpulse.silver.carbon_intensity;

In [ ]:
%sql
SELECT
    rejection_reason,
    COUNT(*) AS rejected_records
FROM gridpulse.silver.carbon_intensity_rejected
GROUP BY rejection_reason
ORDER BY rejected_records DESC;

In [ ]:
%sql
SELECT
    DATE(period_start_utc) AS electricity_date,
    COUNT(*) AS total_periods,
    COUNT(actual_gco2_per_kwh) AS periods_with_actuals,

    ROUND(
        AVG(forecast_gco2_per_kwh),
        2
    ) AS average_forecast,

    ROUND(
        AVG(actual_gco2_per_kwh),
        2
    ) AS average_actual,

    ROUND(
        AVG(absolute_forecast_error_gco2_per_kwh),
        2
    ) AS mean_absolute_error

FROM gridpulse.silver.carbon_intensity

GROUP BY DATE(period_start_utc)
ORDER BY electricity_date;

In [ ]:
%sql
SELECT
    (SELECT COUNT(*)
     FROM gridpulse.bronze.carbon_intensity)
        AS bronze_snapshot_rows,

    (SELECT COUNT(*)
     FROM gridpulse.silver.carbon_intensity)
        AS silver_canonical_rows,

    (SELECT COUNT(*)
     FROM gridpulse.silver.carbon_intensity_rejected)
        AS rejected_rows;

In [ ]:
%sql
SELECT COUNT(*) AS silver_rows_after_rerun
FROM gridpulse.silver.carbon_intensity;